<a href="https://colab.research.google.com/github/damnshah17/Damnshah/blob/main/DL_Assignment3_DaamanShah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install -q transformers torch datasets accelerate bitsandbytes
!pip install -q sentencepiece
!pip install -q scikit-learn nltk
!pip install -q peft
!pip install -q matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.7 MB/s eta 0:00:00


In [12]:
import os

project_path = '/content/drive/MyDrive/DL3-DaamanShah'
os.makedirs(project_path, exist_ok=True)

folders = ['models', 'data', 'checkpoints', 'results', 'logs']
for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print(f"The project Path is: {project_path}")
print(f"The Folders are ")
for folder in folders:
    print(f"/{folder}/")

The project Path is: /content/drive/MyDrive/DL3-DaamanShah
The Folders are 
/models/
/data/
/checkpoints/
/results/
/logs/


In [14]:
def preprocess_lovecraft_text(text):
    start_markers = [
        r'\*\*\* START OF (THE|THIS) PROJECT GUTENBERG EBOOK',
        r'\*\*\*START OF',
        r'Produced by',
        r'Title:',
        r'Author:',
        r'Release Date:'
    ]

    end_markers = [
        r'\*\*\* END OF (THE|THIS) PROJECT GUTENBERG EBOOK',
        r'\*\*\*END OF',
        r'End of the Project'
    ]

    start_pos = 0
    end_pos = len(text)

    for marker in start_markers:
        import re
        match = re.search(marker, text, re.IGNORECASE)
        if match:
            start_pos = match.end()

    for marker in end_markers:
        match = re.search(marker, text, re.IGNORECASE)
        if match:
            end_pos = match.start()

    if start_pos < end_pos:
        text = text[start_pos:end_pos]

    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    text = re.sub(r'\[Illustration:.*?\]', '', text)
    text = re.sub(r'\[Image:.*?\]', '', text)
    text = re.sub(r'\[.*?\]', '', text)

    text = re.sub(r'\n\d+\n', '\n', text)

    return text.strip()

In [20]:
def load_and_preprocess_books(folder_path):
    """
    Load all books from folder and preprocess them
    """
    all_texts = []
    book_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]

    print(f"Found {len(book_files)} text files")

    for book_file in sorted(book_files):
        file_path = os.path.join(folder_path, book_file)
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()

            processed_text = preprocess_lovecraft_text(text)

            if len(processed_text) > 1000:
                all_texts.append(processed_text)
                print(f"Loaded {book_file}: {len(processed_text):,} characters")
            else:
                print(f"Skipped {book_file}: Since it is too short after cleaning")

        except Exception as e:
            print(f"Error loading {book_file}: {e}")

    return all_texts


In [21]:

print("LOADING LOVECRAFT BOOKS")
lovecraft_books_path = '/content/drive/MyDrive/Data-DL3'
print(f"\n The Data is present at : {lovecraft_books_path}")
books = load_and_preprocess_books(lovecraft_books_path)
print(f"\nTotal books loaded: {len(books)}")
print(f"Total words: {sum(len(b.split()) for b in books):,}")

LOADING LOVECRAFT BOOKS

 The Data is present at : /content/drive/MyDrive/Data-DL3
Found 17 text files
Loaded book1.txt: 69,837 characters
Loaded book10.txt: 50,813 characters
Loaded book11.txt: 111,766 characters
Loaded book12.txt: 28,767 characters
Loaded book13.txt: 33,559 characters
Loaded book14.txt: 24,561 characters
Loaded book15.txt: 50,191 characters
Loaded book16.txt: 39,749 characters
Loaded book17.txt: 12,076 characters
Loaded book2.txt: 601,999 characters
Loaded book3.txt: 96,853 characters
Loaded book4.txt: 260,465 characters
Loaded book5.txt: 55,317 characters
Loaded book6.txt: 20,778 characters
Loaded book7.txt: 48,528 characters
Loaded book8.txt: 61,350 characters
Loaded book9.txt: 85,277 characters

Total books loaded: 17
Total words: 279,861


In [ ]:
# ============================================
# 5. LOAD TINYLLAMA MODEL
# ============================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("="*60)
print("LOADING TINYLLAMA MODEL")
print("="*60)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Added pad token as eos token")

print(f"✓ Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# Load model with 4-bit quantization to save memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"✓ Model loaded successfully!")
print(f"✓ Model parameters: {model.num_parameters():,}")

In [22]:
combined_text = "\n".join(books)

print(f"Combined {len(books)} books into one text")
print(f"Total characters: {len(combined_text):,}")
print(f"Total words: {len(combined_text.split()):,}")

# Save the combined text to file for reference
combined_text_path = f"{project_path}/data/combined_lovecraft.txt"
with open(combined_text_path, 'w', encoding='utf-8') as f:
    f.write(combined_text)

print(f"Combined text saved to: {combined_text_path}")


Combined 17 books into one text
Total characters: 1,651,902
Total words: 279,861
Combined text saved to: /content/drive/MyDrive/DL3-DaamanShah/data/combined_lovecraft.txt


In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print("Lets LOad TINYLLAMA MODEL")


model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Added pad token as eos token")

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Lets LOad TINYLLAMA MODEL


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 1,100,048,384
